# NBA 3.1 Feature Engineering: Competitor Drug NBRx Share (Notebook 31)

> **Nov 2026 update:** source switched to `kerendia_laad_based_lrx_v7`, competitor groups re-defined on V7 names (SGLT2i total = all V7 SGLT2 class + Inpefa; MRA = Kerendia + all V7 sMRA groups; HF = Kerendia + Entresto brand/generic + Verquvo + Inpefa + Farxiga + Jardiance). Per-drug SGLT2i shares fixed (they were never computed before; Steglatro was misspelled). Output is one row per HCP x claims cohort (c1-c28), windows ending at each cohort's `lookback_end_date`. Output: `hcp_level_model/dev_pipeline/notebook_output/hcp_features_v7_cohort/competitor_share_features` (test runs go to `_test/`).

**Purpose:** Compute HCP-level Kerendia NBRx market share vs competitors across SGLT2i, MRA, and HF drug classes, using V6 LAAD LRx table with TREATMENT_CATEGORY from the final product mapping.

**Key changes from NBA 3.0:**
- Add Inpefa to SGLT2i competitor set (5 competitors: Farxiga, Invokana, Jardiance, Steglatro, Inpefa)
- Add MRA competitor share (Kerendia / (Kerendia + Spironolactone + Eplerenone))
- Add HF drug share (Kerendia / (Kerendia + Entresto + Verquvo + Inpefa + Farxiga + Jardiance))
- Switch from NPI_NUMBER to PROVIDER_ID (aligns with notebooks 26/27/28)
- Add 1, 3, 6-month windows (currently only 12-month)
- Use step2_fill_missing and step3_lag from 00_step23_function (same as NBA 3.0)

**Data source:** V6 LAAD LRx table (same as NBA 3.0)
**Output:** `kerendia_nba_3_1_volume/notebook_output/hcp_features/competitor_share_features/`

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_step23_function"

In [0]:
# Cell 1: Configuration
import gc
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, expr, concat, substring, trunc

LRX_TABLE = "kerendia_laad_based_lrx_v7"  # Nov 2026: switched from KERENDIA_LAAD_BASED_LRX_V6; V7 groups used as-is
VOLUME_BASE = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
HCP_UNIVERSE_PATH = f"{VOLUME_BASE}/HCP_Universe_without_call_hcps"
COHORTS_PATH = f"{VOLUME_BASE}/cohorts/claims/cohorts.parquet"
OUTPUT_BASE = f"{VOLUME_BASE}/hcp_level_model/dev_pipeline/notebook_output/hcp_features_v7_cohort_aligned"  # same folder as notebooks 26/27/28

HCP_KEY = "PROVIDER_ID"  # Switched from NPI_NUMBER (decision #11)
LAG_WINDOWS = [1, 3, 6, 12]

# SGLT2i competitors: 5 (added Inpefa)
SGLT2I_GROUPS = ["FARXIGA", "INVOKANA", "JARDIANCE", "STEGLATRO", "INPEFA"]  # per-drug share numerators (STEGLATRO typo fixed)
# SGLT2i market (share denominator) = every V7 PRODUCT_CLASS 'SGLT2' group + INPEFA; FARXIGA BRAND/GENERIC are consolidated to FARXIGA
SGLT2I_TOTAL_GROUPS = ["FARXIGA", "INVOKANA", "JARDIANCE", "STEGLATRO", "INPEFA", "BEXAGLIFLOZIN", "BRENZAVVY", "DAPAGLIFLOZIN/METFORMIN H", "GLYXAMBI", "INVOKAMET", "INVOKAMET XR", "QTERN", "SEGLUROMET", "STEGLUJAN", "SYNJARDY", "SYNJARDY XR", "TRIJARDY XR", "XIGDUO XR"]
# MRA competitors: Kerendia vs Spironolactone/Eplerenone
MRA_GROUPS = ["KERENDIA", "SPIRONOLACTONE", "ALDACTONE", "ALDACTAZIDE", "CAROSPIR", "EPLERENONE", "INSPRA"]  # V7 PRODUCT_CLASS 'sMRA' + Kerendia
# HF drug competitors: Kerendia, Entresto, Verquvo, Inpefa, Farxiga, Jardiance
HF_DRUG_GROUPS = ["KERENDIA", "ENTRESTO BRAND", "ENTRESTO GENERIC", "VERQUVO", "INPEFA", "FARXIGA", "JARDIANCE"]

all_competitor_groups = sorted(set(SGLT2I_TOTAL_GROUPS + MRA_GROUPS + HF_DRUG_GROUPS))
print(f"Competitor groups: {len(all_competitor_groups)}")
print(f"SGLT2i: {len(SGLT2I_GROUPS)}, MRA: {len(MRA_GROUPS)}, HF drug: {len(HF_DRUG_GROUPS)}")
from pyspark.storagelevel import StorageLevel

dbutils.widgets.text("test_mode", "false")
TEST_MODE = dbutils.widgets.get("test_mode") == "true"
TEST_HCP_COUNT = 200
# Test runs write to a separate _test folder so they never overwrite full outputs
OUTPUT_PATH = f"{OUTPUT_BASE}/_test/competitor_share_features" if TEST_MODE else f"{OUTPUT_BASE}/competitor_share_features"
print(f"Key: {HCP_KEY}, Windows: {LAG_WINDOWS}")
print(f"Output: {OUTPUT_PATH}")
print(f"Test mode: {TEST_MODE}, Test HCP count: {TEST_HCP_COUNT}")

In [0]:
# Cell 2: Read V6, filter to HCP universe, date processing
# OPTIMIZED: date filter in SQL, broadcast join, cache, minimal count() calls
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID")
if TEST_MODE:
    hcp_universe = hcp_universe.orderBy("PROVIDER_ID").limit(TEST_HCP_COUNT)  # deterministic sample
    print(f"TEST MODE: limiting to {TEST_HCP_COUNT} HCPs")

query = f"""
    SELECT DISTINCT WEEK_ENDING_DATE, PROVIDER_ID, PERIOD_ID, PRODUCT_GROUP, NBRX, NPI, PRI_SPCL_DESC
    from `ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.{LRX_TABLE}
"""
kerendia_claims = spark.sql(query)
kerendia_claims = F.broadcast(hcp_universe).join(kerendia_claims, on="PROVIDER_ID", how="inner")

kerendia_claims = kerendia_claims.withColumn("SVC_DT_M", F.trunc(col("PERIOD_ID"), 'month')).drop('WEEK_ENDING_DATE', 'NPI', 'PRI_SPCL_DESC')
kerendia_claims.persist(StorageLevel.MEMORY_AND_DISK)
materialized_count = kerendia_claims.count()
print(f"Cached V6 (filtered): {materialized_count:,} rows")
index_cols = ["PROVIDER_ID", "SVC_DT_M"]

# Claims cohorts: HCP x cohort skeleton (applied in the Save cell)
df_cohorts = spark.read.parquet(COHORTS_PATH)
print(f"Claims cohorts: {df_cohorts.count()}")
df_hcp_cohorts = hcp_universe.crossJoin(F.broadcast(df_cohorts.select("cohort_id", "prediction_month", "lookback_start_date", "lookback_end_date")))
df_hcp_cohorts = df_hcp_cohorts.withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("cohort_id")))

In [0]:
# Cell 3: Pivot by PRODUCT_GROUP, consolidate FARXIGA, select competitors (exact V7 names), compute totals
from functools import reduce
# Consolidate FARXIGA (V7: FARXIGA BRAND + FARXIGA GENERIC -> FARXIGA), same as NBA 3.0
df_category_claims = kerendia_claims.withColumn("PRODUCT_GROUP", when(col("PRODUCT_GROUP").like("%FARXIGA%"), "FARXIGA").otherwise(col("PRODUCT_GROUP")))
df_category_claims = df_category_claims.withColumn("PRODUCT_GROUP", when((col("PRODUCT_GROUP").isNull()) | (col("PRODUCT_GROUP") == 'NULL'), 'UNKNOWN').otherwise(col("PRODUCT_GROUP")))

# Pivot only the competitor groups (exact names), sum NBRX
df_category = (df_category_claims.filter(col("PRODUCT_GROUP").isin(all_competitor_groups))
               .groupBy(index_cols).pivot("PRODUCT_GROUP", all_competitor_groups).agg(F.sum("NBRX").alias("SUM_NBRX")))

# Clean column names (replace spaces, special chars for Delta compatibility)
def clean_col_name(c):
    return c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()

df_category_new = df_category.select([F.col(f"`{c}`").alias(clean_col_name(c)) if c not in index_cols else col(c) for c in df_category.columns])

# Market totals (null counts as 0)
def total_of(groups):
    return reduce(lambda a, b: a + b, [F.coalesce(col(clean_col_name(g)), F.lit(0)) for g in groups])

df_category_new = (df_category_new
    .withColumn("TOTAL_NBRX_CLAIMS_SGLT2I", total_of(SGLT2I_TOTAL_GROUPS))
    .withColumn("TOTAL_NBRX_CLAIMS_MRA", total_of(MRA_GROUPS))
    .withColumn("TOTAL_NBRX_CLAIMS_HF_DRUG", total_of(HF_DRUG_GROUPS)))

print(f"Competitor data: {df_category_new.count():,} rows, {len(df_category_new.columns)} cols")

In [0]:
# Cell 4: step2_fill_missing and step3_lag with [1, 3, 6, 12]
# OPTIMIZED: removed count() call
df_nbrx_shares = step2_fill_missing(df_category_new, key_col="PROVIDER_ID", date_col="SVC_DT_M")
keep_cols = ["PROVIDER_ID", "SVC_DT_M"]
cols_to_drop = [c for c in df_nbrx_shares.columns if c not in keep_cols]
df_nbrx_feats = step3_lag(df_nbrx_shares, key_col="PROVIDER_ID", date_col="SVC_DT_M", last_i_month_list=LAG_WINDOWS)
del df_nbrx_shares; gc.collect()
df_nbrx_feats = df_nbrx_feats.drop(*cols_to_drop).filter(df_nbrx_feats.PROVIDER_ID.isNotNull()).fillna(0)
print(f"After step3_lag: {len(df_nbrx_feats.columns)} cols")

In [0]:
# Cell 5: Compute share features for all 3 groups x 4 windows (exact column names)
# SGLT2i shares: PERC_{DRUG}_NBRX_SHARE_{WINDOW}M = drug / total_SGLT2i
# MRA share:     PERC_KERENDIA_MRA_SHARE_{WINDOW}M = Kerendia / total_MRA
# HF drug share: PERC_KERENDIA_HF_DRUG_SHARE_{WINDOW}M = Kerendia / total_HF_drug
KERENDIA_COL = clean_col_name("KERENDIA")
share_exprs = []
for lag_m in LAG_WINDOWS:
    sfx = f"_IN_LAST_{lag_m}_MONTH"
    sg_total, mra_total, hf_total = f"TOTAL_NBRX_CLAIMS_SGLT2I{sfx}", f"TOTAL_NBRX_CLAIMS_MRA{sfx}", f"TOTAL_NBRX_CLAIMS_HF_DRUG{sfx}"
    for drug in SGLT2I_GROUPS:
        d = clean_col_name(drug)
        share_exprs.append(when(col(sg_total) != 0, col(d + sfx) / col(sg_total)).otherwise(0).alias(f"PERC_{d}_NBRX_SHARE_{lag_m}M"))
    share_exprs.append(when(col(mra_total) != 0, col(KERENDIA_COL + sfx) / col(mra_total)).otherwise(0).alias(f"PERC_KERENDIA_MRA_SHARE_{lag_m}M"))
    share_exprs.append(when(col(hf_total) != 0, col(KERENDIA_COL + sfx) / col(hf_total)).otherwise(0).alias(f"PERC_KERENDIA_HF_DRUG_SHARE_{lag_m}M"))

df_nbrx_feats = df_nbrx_feats.select("*", *share_exprs)
print(f"Share features computed: {len(share_exprs)}. Total columns: {len(df_nbrx_feats.columns)}")

In [0]:
# Cell 6: Filter and save
# OPTIMIZED: removed count() call
df_nbrx_feats = df_nbrx_feats.filter(F.to_date('SVC_DT_M') >= '2022-11-01')

# Apply claims cohort windows: one row per HCP x cohort, taken at feature month = lookback_end_date.
# step3_lag windows include the current month, so *_IN_LAST_12_MONTH = lookback_start_date..lookback_end_date.
cohort_keys = df_hcp_cohorts.select("HCP_COHORT_ID", "PROVIDER_ID", "cohort_id", "prediction_month", "lookback_start_date", "lookback_end_date")
df_nbrx_feats = cohort_keys.join(df_nbrx_feats.withColumnRenamed("SVC_DT_M", "lookback_end_date"), on=["PROVIDER_ID", "lookback_end_date"], how="left").fillna(0)
curr_date = str(dt.today().strftime('%Y_%m_%d'))
df_nbrx_feats = df_nbrx_feats.withColumn("PULL_DT", F.lit(curr_date))
print(f"Saving to: {OUTPUT_PATH}")
df_nbrx_feats.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
print(f"Notebook 31 (Competitor Share) complete. Saved to NBA 3.1 volume.")